# 10주차. 머신러닝 모델 서비스 구조

| 항목 | 내용 |
|---|---|
| 이름 | 이성민 |
| 학과 | 소프트웨어융합과 |
| 학년 | 2학년 |
| 학번 | 2151050 |

이 노트북은 수업 주차 흐름을 참고해 우리 방식으로 재구성한 실행형 설명 자료입니다. 외부 서비스 호출 없이 실행되도록 작은 샘플 데이터를 사용합니다.

## 모델 학습과 저장
웹 서비스에서는 모델 학습과 예측 API를 분리하는 것이 일반적입니다. 여기서는 작은 규칙 기반 분류기를 학습 객체처럼 저장합니다.

In [1]:
import pickle
import pandas as pd
from pathlib import Path

train = pd.DataFrame([
    {"petal": 1.4, "sepal": 5.1, "label": "setosa"},
    {"petal": 4.7, "sepal": 6.3, "label": "versicolor"},
    {"petal": 5.8, "sepal": 7.1, "label": "virginica"},
])

model = {
    "thresholds": {"setosa_max": 2.0, "versicolor_max": 5.2},
    "labels": ["setosa", "versicolor", "virginica"],
}

model_path = Path("week10_model.pkl")
model_path.write_bytes(pickle.dumps(model))

def predict_species(petal):
    loaded = pickle.loads(model_path.read_bytes())
    if petal <= loaded["thresholds"]["setosa_max"]:
        return "setosa"
    if petal <= loaded["thresholds"]["versicolor_max"]:
        return "versicolor"
    return "virginica"

assert predict_species(1.5) == "setosa"
assert predict_species(5.5) == "virginica"
train

,petal,sepal,label
0,1.4,5.1,setosa
1,4.7,6.3,versicolor
2,5.8,7.1,virginica


## 서비스 함수
FastAPI나 Gradio가 붙더라도 핵심은 입력값을 검증하고 예측 결과를 반환하는 함수입니다.

In [2]:
def service_predict(petal, sepal):
    if petal <= 0 or sepal <= 0:
        raise ValueError("measurements must be positive")
    label = predict_species(petal)
    return {"petal": petal, "sepal": sepal, "prediction": label}

payload = service_predict(4.8, 6.1)
assert payload["prediction"] == "versicolor"
payload

{'petal': 4.8, 'sepal': 6.1, 'prediction': 'versicolor'}